# Prompt Evaluation Pipeline —— 框架总览

这个 notebook 实现了一套完整的 **prompt evaluation(prompt 效果评测)** 流程,一共分四个阶段:

1. **准备 client + 自动生成数据集**:初始化 Anthropic SDK 的 client,然后用一次 LLM 调用自动生成一批测试用例(dataset),存入 `dataset.json`。
2. **定义评测所需的三个角色函数**:
   - `run_prompt`:被测试的"选手"——用某个 prompt 去解决一条 task,拿到 `output`
   - `grade_by_model`:"评委"——用另一次 LLM 调用给 `output` 打分(1-10),同时要求先给出 strengths / weaknesses / reasoning
   - `run_test_case`:把"选手"和"评委"串起来,处理单条 task
3. **批量跑评测**:`run_eval` 遍历整个 dataset,对每条 task 都跑一遍 `run_test_case`,最后算平均分
4. **加载、执行、打印结果**:从 `dataset.json` 读回数据集,触发 `run_eval`,打印详细的评测结果

## 这套代码是不是典型的 SDK / API 使用模板?

是的。这是 Anthropic Messages API 最典型的用法组合,核心就三件事:
- 所有对话都通过同一个 API `client.messages.create(model=..., max_tokens=..., messages=..., system=..., temperature=..., stop_sequences=...)` 完成,自己在 Python 里维护 `messages` 列表(role 必须 user/assistant 交替,API 本身是无状态的)
- 大量使用了 **prefill**(assistant 消息预填充)技巧:手动在 assistant 回合开头写 `"```json"`,强迫模型直接接着往下写 JSON,不带任何开场白解释
- 配合 `stop_sequences=["```"]` 截断输出,拿到干净的 JSON 文本

下面的对话回复里会对这些典型 API 用法逐一举例说明。

In [1]:
# Load env variables and create client
from dotenv import load_dotenv  # 从 .env 文件读取 ANTHROPIC_API_KEY
from anthropic import Anthropic  # Anthropic 官方 SDK 的核心 client 类

load_dotenv()  # 把 .env 里的 environment variable 加载进 os.environ

# client = Anthropic()  # 标准写法:zero-parameter 构造,自动从 ANTHROPIC_API_KEY 这个 environment variable 取 key

import httpx
# verify=False 关闭 SSL 证书校验(公司内网/代理环境常见的临时绕过方式,生产环境不建议这样用)
client = Anthropic(http_client=httpx.Client(verify=False))
model = "claude-haiku-4-5"  # 用最便宜/最快的模型,因为这里只是跑评测流程,不追求最高质量

In [2]:
# Helper functions
def add_user_message(messages, text):
    # 往 messages 这个 list 末尾追加一条 user role 的 message
    # content 直接传 string 即可,SDK 会自动包装成 Messages API 要求的格式
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    # 往 messages 这个 list 末尾追加一条 assistant role 的 message
    # 注意:这个函数既用来记录模型的真实回复,也用来做 prefill(见下方 chat() 的调用方式)
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    # 封装 client.messages.create() 的核心调用,统一管理常用 parameter
    params = {
        "model": model,
        "max_tokens": 1000,  # 限制单次回复最多生成的 token 数,超过会被截断(stop_reason 变成 max_tokens)
        "messages": messages,
        "temperature": temperature,  # 采样随机性:0 更确定,1 更多样(这里默认 1.0)
        "stop_sequences": stop_sequences,  # 遇到指定 string 就立刻停止生成,常配合 prefill 使用
    }

    if system:
        params["system"] = system  # system 是独立 parameter,不放在 messages 这个 list 里

    message = client.messages.create(**params)
    # message.content 是一个 list(可能包含 text/tool_use 等多种 block),这里只取第一个 text block 的内容
    return message.content[0].text

In [3]:
# Function to generate a new dataset
import json


def generate_dataset():
    # 这个 prompt 要求模型"生成评测用的任务列表",并在 prompt 里给了输出格式的例子(few-shot 式引导)
    # 新增 solution_criteria 字段:给每个 task 附上几条具体、可核查的评分标准,
    # 目的是让下面 grade_by_model() 打分时有明确依据,而不是纯靠整体印象判断
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python code using the pandas or numpy libraries. Generate an array of JSON objects,
each representing a beginner-level task that can be solved with a few lines of pandas or numpy code.

Example output:
```json
[
    {
        "task": "Given a DataFrame with columns 'name' and 'age', return only the rows where age is greater than 30",
        "format": "pandas",
        "solution_criteria": [
            "Filters using boolean indexing (e.g. df[df['age'] > 30]), not iterrows() or apply()",
            "Returns a DataFrame, not a Series or list",
            "Does not mutate the original DataFrame"
        ]
    },
    {
        "task": "Given a 1D numpy array of numbers, return a new array with each element squared",
        "format": "numpy",
        "solution_criteria": [
            "Uses vectorized operations (e.g. arr ** 2), not a Python for-loop",
            "Returns a new array without modifying the input array in place"
        ]
    },
    ...additional
]
```

* Focus on the most common beginner operations:
  - pandas: filtering rows by a condition, selecting columns, groupby + mean/sum, sorting,
    renaming columns, filling missing values with fillna.
  - numpy: element-wise math, array reductions (sum/mean/max along an axis), boolean masking,
    reshaping, simple broadcasting.
* Each "task" must be a self-contained one-sentence description that names the concrete input
  (e.g. "a DataFrame with columns X and Y", "a 2D numpy array of shape (3, 4)") and the expected output.
* Each "solution_criteria" must be an array of 2-3 concrete, checkable conditions a correct solution
  must satisfy (e.g. which method/approach to use, what edge cases to handle, what must NOT change).
  Avoid vague criteria like "the solution must be correct".
* Keep every task solvable in a single short function or 1-3 lines of code. No file I/O, no plotting,
  no external data sources.

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    # 关键的 prefill 技巧:手动往 assistant 回合塞入 "```json" 作为开头
    # 这样模型的"续写"就会直接从 JSON 内容开始,不会先说"好的,这是您要的数据集:"这类客套话
    add_assistant_message(messages, "```json")
    # stop_sequences=["```"] 让生成到遇到结尾的 ``` 就停止,这样拿到的是干净的 JSON 主体(不含首尾的代码块标记)
    text = chat(messages, stop_sequences=["```"])
    print(text)
    # 用 JSONDecoder().raw_decode 而不是 json.loads,是因为 raw_decode 允许 string 末尾有多余内容也不报错
    # (对比下面 grade_by_model 里踩坑用的 json.loads,会在有多余内容时抛 JSONDecodeError)
    return json.JSONDecoder().raw_decode(text.strip())[0]

In [4]:
# Generate the dataset and write it to 'dataset.json'
dataset = generate_dataset()  # 调用一次 LLM,拿到 3 条任务组成的 list
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)  # 落盘保存,后面评测时直接从文件读取,不用每次都重新生成(省钱、结果可复现)


[
    {
        "task": "Given a DataFrame with columns 'product', 'quantity', and 'price', return a new DataFrame containing only rows where quantity is greater than 5, sorted by price in descending order",
        "format": "pandas",
        "solution_criteria": [
            "Filters using boolean indexing (e.g. df[df['quantity'] > 5]), not iterrows() or apply()",
            "Sorts using .sort_values() method with ascending=False parameter",
            "Does not mutate the original DataFrame"
        ]
    },
    {
        "task": "Given a 2D numpy array of shape (4, 3) containing numerical values, return a 1D array containing the mean of each row",
        "format": "numpy",
        "solution_criteria": [
            "Uses .mean() method with axis=1 parameter or np.mean() with axis argument, not a Python for-loop",
            "Returns a 1D array of length 4",
            "Does not mutate the input array"
        ]
    },
    {
        "task": "Given a DataFrame with columns 'de

In [5]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    # "评委" prompt:把原始任务 test_case["task"]、solution_criteria 和"选手"的回答 output 一起塞进去,让模型当裁判打分
    # 新增 <criteria> block:把 dataset generation 阶段生成的 solution_criteria 传进来,
    # 让模型对照这几条具体标准逐条核对,而不是纯凭整体印象判断"好不好"
    eval_prompt = f"""
You are an expert Python data engineer specializing in pandas and numpy. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution Criteria:
<criteria>
{test_case["solution_criteria"]}
</criteria>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Grade the solution based on whether it satisfies each item in the criteria above, not just whether it
"looks reasonable". Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """
    # 关键设计:先要求模型给出 strengths/weaknesses/reasoning,最后才给 score
    # 这样模型被迫先"想清楚理由"再打分,不会不假思索地都打中间分(比如总是打 6 分)——这是文章开头提到的核心 insight

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")  # 同样用 prefill 强制模型直接输出 JSON,没有寒暄
    eval_text = chat(messages, stop_sequences=["```"])
    # 踩坑点:这里用了 json.loads 而不是上面 generate_dataset() 用的 raw_decode
    # 如果模型在 JSON 后面多输出了任何字符(比如多余的换行或说明文字),json.loads 会抛 JSONDecodeError("Extra data")
    # 这正是下方 run_eval 报错的原因,修复方式是改成 json.JSONDecoder().raw_decode(eval_text.strip())[0]
    return json.JSONDecoder().raw_decode(eval_text.strip())[0]


In [6]:
# Passes a test case into Claude
def run_prompt(test_case):
    # "选手" prompt:直接把任务描述丢给模型,不做任何格式限制,拿到模型的自然语言/代码混合回答
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)  # 注意这里没有 prefill、没有 stop_sequences——就是普通的一次问答
    return output

In [7]:
# Function to execute a single test case and grade the output
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)  # 第一次 LLM 调用:让"选手"模型解题

    model_grade = grade_by_model(test_case, output)  # 第二次 LLM 调用:让"评委"模型打分
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    # 把这条 task 的完整信息打包返回,方便后面汇总和打印
    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [8]:
from statistics import mean


def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)  # 每条 task 都要打 2 次 LLM 调用(1 次解题 + 1 次打分)
        results.append(result)

    average_score = mean([result["score"] for result in results])  # 算所有 task 的平均分,作为这个 prompt 的整体质量指标
    print(f"Average score: {average_score}")

    return results

In [9]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)  # 从磁盘读回之前生成好的评测数据集,不用重新调 LLM 生成

results = run_eval(dataset)  # 触发整个评测流程:对 dataset 里每条 task 跑一遍"解题 + 打分"
# 下方报错 JSONDecodeError: Extra data 就是 grade_by_model() 里 json.loads() 踩坑的实际案例
# (模型输出的 JSON 后面带了多余字符,json.loads 无法容忍,应换成 json.JSONDecoder().raw_decode())

Average score: 8


In [10]:
print(json.dumps(results, indent=2))  # 把所有 task 的 output/score/reasoning 完整打印出来,便于人工复核评测结果

[
  {
    "output": "# Solution to Filter and Sort DataFrame\n\nHere's a complete solution with explanation:\n\n```python\nimport pandas as pd\n\ndef filter_and_sort_dataframe(df):\n    \"\"\"\n    Filter DataFrame for quantity > 5 and sort by price descending.\n    \n    Parameters:\n    df (pd.DataFrame): DataFrame with 'product', 'quantity', and 'price' columns\n    \n    Returns:\n    pd.DataFrame: Filtered and sorted DataFrame\n    \"\"\"\n    # Filter rows where quantity > 5\n    filtered_df = df[df['quantity'] > 5]\n    \n    # Sort by price in descending order\n    result_df = filtered_df.sort_values('price', ascending=False)\n    \n    return result_df\n\n\n# Example usage:\nif __name__ == \"__main__\":\n    # Create sample DataFrame\n    data = {\n        'product': ['Apple', 'Banana', 'Orange', 'Mango', 'Grape', 'Pear'],\n        'quantity': [10, 3, 8, 5, 7, 12],\n        'price': [1.5, 0.5, 2.0, 3.5, 1.2, 2.5]\n    }\n    \n    df = pd.DataFrame(data)\n    \n    print(\"Ori